In [1]:
!pip install shap  # if not already installed

import torch
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

ERROR: Invalid requirement: '#'

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
Matplotlib is building the font cache; this may take a moment.


In [ ]:
from 03_lstm_forecasting import LSTMForecaster
# Load your trained model from Phase 3
model = LSTMForecaster(input_size=X_seq.shape[2], hidden_size=64, num_layers=2)
model.load_state_dict(torch.load("models/lstm_best.pth"))
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Load the engineered dataset
df = pd.read_csv("data/processed/tsla_engineered_final.csv", index_col='Date', parse_dates=True)

ModuleNotFoundError: No module named 'lstm_forecaster'

In [ ]:
# Use a subset of training data as background (very important)
background_size = 200   # Keep small due to computation cost
background_data = X_train[:background_size]   # from Phase 3

print(f"Background shape: {background_data.shape}")

In [ ]:
# Define prediction function for SHAP
def model_predict(data):
    model.eval()
    with torch.no_grad():
        data_tensor = torch.tensor(data, dtype=torch.float32).to(device)
        if len(data_tensor.shape) == 2:   # if single sample
            data_tensor = data_tensor.unsqueeze(0)
        predictions = model(data_tensor).cpu().numpy()
    return predictions

# Create Kernel Explainer (works with LSTM via flattened input or custom wrapper)
explainer = shap.KernelExplainer(model_predict, background_data)

print("Explainer created successfully!")

In [ ]:
# Compute explanations on test set (use small sample first!)
test_sample_size = 50   # Start small! Increase later
X_test_sample = X_test[:test_sample_size]

shap_values = []
confidences = []

print("Computing TimeSHAP explanations... (this may take 10-30 minutes)")

for i in tqdm(range(len(X_test_sample))):
    instance = X_test_sample[i:i+1]
    
    try:
        shap_value = explainer.shap_values(instance, nsamples=100)  # nsamples = accuracy vs speed tradeoff
        shap_values.append(shap_value)
        
        # Simple confidence calculation (based on magnitude of explanations)
        importance = np.abs(shap_value).mean()
        confidence = 1.0 - np.tanh(importance * 5)   # invert and normalize
        confidences.append(max(0.1, confidence))
        
    except Exception as e:
        print(f"Error at index {i}: {e}")
        confidences.append(0.5)

# Save results
np.save("results/shap_values.npy", np.array(shap_values))
df_test = df.iloc[-len(X_test):].copy()
df_test = df_test.iloc[:test_sample_size].copy()
df_test['confidence_score'] = confidences

print("Confidence scores generated!")
print(df_test['confidence_score'].describe())

In [ ]:
# Plot confidence over time
plt.figure(figsize=(12, 6))
plt.plot(df_test.index, df_test['confidence_score'], label='Confidence Score', color='purple')
plt.title('Daily Model Confidence Score (from TimeSHAP)')
plt.ylabel('Confidence')
plt.xlabel('Date')
plt.legend()
plt.grid(True)
plt.savefig("results/confidence_scores.png")
plt.show()

# Save final dataset with confidence
df_test.to_csv("data/processed/tsla_with_confidence.csv")